In [6]:
### 文本擷取

import fitz  # PyMuPDF
import re


def step1_extract_text(pdf_path):
    """
    從 PDF 中提取文字，並進行初步的格式清理。
    """
    try:
        # 開啟 PDF 檔案
        doc = fitz.open(pdf_path)
        print(f"--- 檔案讀取成功：{pdf_path} ---")
        print(f"總頁數: {len(doc)}")
        
        extracted_data = []

        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            # 提取文字
            raw_text = page.get_text("text")
            
            # 初步清理：移除多餘的連續空白、統一換行符
            clean_text = re.sub(r'\n\s*\n', '\n\n', raw_text) # 保持段落感
            clean_text = clean_text.strip()
            
            # 儲存結果（包含頁碼資訊，這對後續 RAG 引用非常重要）
            extracted_data.append({
                "page": page_num + 1,
                "content": clean_text
            })
            
            # 預覽前兩頁
            if page_num < 2:
                print(f"\n[第 {page_num + 1} 頁預覽]:")
                print(clean_text[:300] + "...") 
                print("-" * 30)

        doc.close()
        return extracted_data

    except Exception as e:
        print(f"讀取失敗：{e}")
        return None

# --- 執行處 ---
# 請將 'HIWIN_Catalog.pdf' 換成你實際的檔案路徑

raw_pages = step1_extract_text(R"../data/上銀滾珠螺桿.pdf")

--- 檔案讀取成功：../data/上銀滾珠螺桿.pdf ---
總頁數: 189

[第 1 頁預覽]:
www.hiwin.com.tw
Ballscrews
Technical Information
滾珠螺桿
技術手冊...
------------------------------

[第 2 頁預覽]:
高速化 
高精度 
複合化 
環保 
生活化
2013年台灣精品金質獎
交叉滾柱軸承
Crossed Roller Bearings
2006年台灣精品銀質獎 
2007年中小企業創新研究獎 
直驅式定位平台
Torque Motor 
Direct drive Motor
2012, 2011, 2009, 2008, 2005年
台灣精品金質獎
2006, 2001,1993年台灣精品銀質獎
滾珠螺桿 Ballscrew
精密研磨/精密轉造
• 高速化 (高 Dm-N 值 /Super S 系列)
• 重負荷滾珠螺桿-全電式射出成型機
• E2 環保潤滑模組
• R1 螺帽旋轉式
• C1...
------------------------------


In [7]:
###文本擷取確認存檔

def save_extraction_to_file(extracted_data, output_filename="HIWIN_extraction_check.md"):
    """
    將擷取到的文字資料存成 Markdown 檔案，並自動計算字數。
    """
    

    try:
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write("# HIWIN 型錄文字擷取結果檢視\n\n")
            f.write(f"**總計擷取頁數:** {len(extracted_data)}\n\n")
            f.write("---\n\n")

            for item in extracted_data:
                # 使用 .get() 確保如果找不到鍵也不會當機，並現場計算字數
                page_no = item.get('page', '未知')
                content = item.get('content', '')
                char_count = len(content)
                
                f.write(f"## 第 {page_no} 頁\n")
                f.write(f"**本頁字數:** {char_count} 字\n\n")
                f.write("### 內容摘要:\n")
                f.write("```text\n")
                f.write(content if content else "[此頁無文字內容]")
                f.write("\n```\n")
                f.write("\n---\n\n")
        
        print(f"檢查檔案已成功生成：{output_filename}")

    except Exception as e:
        print(f"儲存失敗，原因：{e}")

# --- 執行處 ---
# 請確保這裡傳入的是你上一個步驟得到的列表變數
#save_extraction_to_file(raw_pages)


In [8]:
### 文本清理確認 --> 頁清洗、段落清洗

import re

def is_useful_page(content):
    # 將內容拆成行，並過濾掉空行
    lines = [l.strip() for l in content.split('\n') if l.strip()]
    if not lines: return False
    
    # --- 1. 核心指標計算 ---
    # 統計有多少行「看起來像數據列」
    # 特徵：數字多、長度短、沒有句號
    data_line_count = 0
    for line in lines:
        # 移除空格後，計算數字佔比
        no_space_line = line.replace(" ", "")
        if not no_space_line: continue
        
        digits = len(re.findall(r'\d', no_space_line))
        digit_ratio = digits / len(no_space_line)
        
        # 如果一行內超過 60% 是數字，或者是像 "16-4B2" 這種型號格式
        if digit_ratio > 0.6 or re.search(r'\d+-\d+', line):
            data_line_count += 1

    # --- 2. 判斷邏輯 ---
    # 判斷 A：數據行比例。如果超過一半的行都是數據，這就是表格頁
    data_density = data_line_count / len(lines)
    
    # 判斷 B：是否有完整論述。說明書一定有「。」或「：」
    has_description = content.count("。") >= 2 or content.count(":") >= 2

    # 如果數據密度高 ( > 40%) 且 缺乏完整論述 ( < 2 個句號)
    # 這種就是你提供的這種規格表，直接移除
    if data_density > 0.4 and not has_description:
        return False
        
    # 判斷 C：極端字數檢查 (針對你範例中的 3288 字)
    # 如果字數極多但句號極少，通常是 PDF 轉檔產生的表格碎片堆疊
    if len(content) > 1000 and content.count("。") < 3:
        return False

    return True

def is_text_description(text):
    """
    更聰明的判定：這段話像不像『人的語言』？
    """
    t = text.strip()
    if len(t) < 15: return False # 太短的通常是噪音
    
   # 提取特徵
    punctuation = len(re.findall(r'[，。：；（）()、]', t)) # 提取"標點符號"類數據
    has_punctuation = punctuation > 0 # 檢查"標點符號"密度
    digits = len(re.findall(r'\d', t))# 提取"數字"類數據
    text_ratio = (len(t) - digits) / len(t) # 計算數字與英文字母的混合程度
    spaces_pattern = len(re.findall(r'\s{2,}', t)) #偵測是否有 2 個以上的連續空格，這通常是表格欄位的間距
    

    # --- 邏輯判斷 ---    
    if spaces_pattern >= 3 and not has_punctuation: # 空格多且沒標點 (典型的表格殘影)
        return False
    if text_ratio < 0.2: #數字比例極高 (純規格數據)
        return False
    if has_punctuation and text_ratio > 0.35:# 只要有標點且文字佔比尚可，就視為說明文字
        return True
    if text_ratio > 0.7: # 標題保留：純文字比例極高
        return True
    
    return True # 預設保留


def step2_dual_layer_clean(extracted_data):
    """
    整合流程：先過濾頁，再過濾段落。
    """
    refined_docs = []
    for item in extracted_data:
        content = item['content']
        
        # --- 第一關：頁面過濾 ---
        if not is_useful_page(content):
            continue # 跳過純數據頁
            
        # --- 第二關：段落過濾 ---
        paragraphs = content.split('\n\n') 
        filtered_paragraphs = [p.strip() for p in paragraphs if is_text_description(p)]
        
        clean_text = "\n\n".join(filtered_paragraphs)
        if clean_text:
            refined_docs.append({"page": item['page'], "content": clean_text})
            
    return refined_docs

# --- 執行清理 ---
purified_data = step2_dual_layer_clean(raw_pages)
save_extraction_to_file(purified_data)

檢查檔案已成功生成：HIWIN_extraction_check.md


In [9]:
###切片工程，chuncks可視化

import re

def semantic_chunking_optimized(purified_data, chunk_size=600, overlap=100):
    chunks = []
    min_chunk_size = 100 
    
    def fix_special_chars(text):
        """
        修正會觸發 Markdown 刪除線的波浪號，並清理 PDF 雜字。
        """
        # 將所有波浪號轉義，確保 Markdown 不會誤判為刪除線
        text = text.replace("~", "\~")
        text = text.replace("~~", "~")
        # 順便把一些奇怪的 PDF 符號換成人類好讀的符號
        text = text.replace("■", "●").replace("□", "○")
        return text

    # ---判斷內容是否為數字垃圾 ---
    def is_garbage_chunk(content):
        # 1. 移除固定編號與所有空格，計算「純淨密度」
        temp_text = re.sub(r'S99TC13-\d+\s\d+-', '', content).strip()
        no_space_text = temp_text.replace(" ", "").replace("\n", "")
        
        if not no_space_text: return True

        # 2. 數據特徵計算 (包含數字、點、負號)
        data_chars = len(re.findall(r'[\d\.\-]', no_space_text))
        # 使用不含空格的長度作為分母，這會讓座標軸的 ratio 飆升
        ratio = data_chars / len(no_space_text)
        
        # 3. 語義特徵偵測
        # 真正的技術說明幾乎一定會包含「。 」或常見中文字
        has_meaningful_connectors = any(word in temp_text for word in ["。 ", "是", "為", "的", "包含", "於"])
        
        # 4. 判斷標準
        # 狀況 A：數字與符號比例極高 (> 75%)
        if ratio > 0.75 and not has_meaningful_connectors:
            return True
            
        # 狀況 B：含有連續的座標軸特徵 (例如出現 3 次以上的 "-數字")
        coord_patterns = len(re.findall(r'-\d+', temp_text))
        if coord_patterns >= 5 and not has_meaningful_connectors:
            return True
            
        # 狀況 C：字數很多但完全沒有中文結尾
        if len(temp_text) > 50 and not has_meaningful_connectors and ratio > 0.5:
            return True
                    
        return False

    for item in purified_data:
        # 在每一頁開始處理前，先做基本的符號修復
        text = fix_special_chars(item['content'])
        page_num = item['page']
        start = 0
        
        while start < len(text):
            remaining_len = len(text) - start
            
            # --- 核心優化邏輯：碎片處理 ---
            if remaining_len < (chunk_size + min_chunk_size) / 2:
                chunk_content = text[start:].strip()
                
                # 攔截垃圾數據
                if is_garbage_chunk(chunk_content):
                    break # 是垃圾就不合併也不處理，直接跳過

                if chunks and chunks[-1]["metadata"]["source_page"] == page_num:
                    chunks[-1]["content"] += "\n\n" + chunk_content
                elif len(chunk_content) > 20: 
                    chunks.append({
                        "content": chunk_content,
                        "metadata": {"source_page": page_num, "type": "technical_manual"}
                    })
                break 

            # --- 正常切片邏輯 ---
            end = start + chunk_size
            if end < len(text):
                last_punctuation = text.rfind('。', start, end + 80)
                if last_punctuation != -1 and last_punctuation > start + (chunk_size // 2):
                    end = last_punctuation + 1
            
            chunk_content = text[start:end].strip()
            
            # 在正式存入前攔截垃圾
            if len(chunk_content) > 30 and not is_garbage_chunk(chunk_content):
                chunks.append({
                    "content": chunk_content,
                    "metadata": {"source_page": page_num, "type": "technical_manual"}
                })
            
            start = end - overlap
            
    print(f"切片完成！共產生 {len(chunks)} 個語義區塊。")
    return chunks

def save_chunks_to_markdown(chunks, filename="HIWIN_chunking_visualization.md"):
    def clean_line_breaks(text):
        """
        將被 PDF 強行切斷的行接起來，但保留真正的段落。
        """
        # 邏輯：如果一行結尾不是句號、冒號、問號，就代表這句話還沒講完，把換行刪掉
        import re
        # 匹配「非結尾標點」+「換行符號」
        fixed_text = re.sub(r'([^。：？！])\n', r'\1', text)
        
        # 處理多餘的空格（PDF 轉檔常出現）
        fixed_text = re.sub(r' +', ' ', fixed_text)
        
        return fixed_text.strip()
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write("# 語義切片可視化報告\n")
        f.write(f"**總切片數**: {len(chunks)} 個\n\n")
        f.write("---\n\n")
        
        for i, chunk in enumerate(chunks):
            cleaned_content = clean_line_breaks(chunk['content'])
            
            f.write(f"##Chunk #{i+1}\n")
            # 寫入 Metadata 資訊
            source = chunk['metadata'].get('source_page', '未知')
            f.write(f"- **來源頁碼**: 第 {source} 頁\n")
            f.write(f"- **字數統計**: {len(cleaned_content)} 字\n")
            f.write(f"- **資料類型**: {chunk['metadata'].get('type', 'N/A')}\n\n")
            
            # 寫入內容，使用引用區塊增加可讀性
            f.write("> ### 內容區塊:\n")
            f.write("> " + cleaned_content.replace("\n", "\n> ") + "\n\n")
            
            # 畫一條分隔線
            f.write("---\n\n")
            
    print(f"可視化檔案已儲存至: {filename}")


# --- 執行 ---
final_chunks = semantic_chunking_optimized(purified_data)
# --- 執行儲存 ---
save_chunks_to_markdown(final_chunks)

切片完成！共產生 87 個語義區塊。
可視化檔案已儲存至: HIWIN_chunking_visualization.md


<>:14: SyntaxWarning: invalid escape sequence '\~'
<>:14: SyntaxWarning: invalid escape sequence '\~'
C:\Users\e11338\AppData\Local\Temp\ipykernel_27744\3918021275.py:14: SyntaxWarning: invalid escape sequence '\~'
  text = text.replace("~", "\~")


In [10]:
import json

# 將切片結果儲存為 JSON，確保 ensure_ascii=False 以正常顯示中文
with open("HIWIN_final_chunks.json", "w", encoding="utf-8") as f:
    json.dump(final_chunks, f, ensure_ascii=False, indent=4)

print("切片資料已儲存為 HIWIN_final_chunks.json")

切片資料已儲存為 HIWIN_final_chunks.json
